# AI Music OS — Qwen Voice Module

This notebook removes the previous fake voice-cloning implementation. It does **not** pretend that `facebook/mms-tts-eng` can clone an uploaded voice. A reference sample is stored persistently, and the generation backend must explicitly support speaker conditioning.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import os, sys, subprocess, time, shutil

ROOT = Path('/content/drive/MyDrive/AI_Music_OS')
VOICE_BANK = ROOT/'voice_bank'
OUT = ROOT/'outputs'/'qwen_voice'
CACHE = ROOT/'cache'/'huggingface'
for p in (VOICE_BANK, OUT, CACHE):
    p.mkdir(parents=True, exist_ok=True)

os.environ['HF_HOME'] = str(CACHE)
os.environ['HUGGINGFACE_HUB_CACHE'] = str(CACHE/'hub')
os.environ['TRANSFORMERS_CACHE'] = str(CACHE/'transformers')

print('Persistent folders ready:', ROOT)


In [ ]:
# Install only generic UI/audio tooling here. No fake MMS-TTS fallback.
subprocess.run([sys.executable,'-m','pip','install','-U','gradio>=5,<7','soundfile','numpy'], check=True)


In [ ]:
import gradio as gr
import soundfile as sf
from pathlib import Path
import shutil, time

def save_voice(sample, profile_name):
    if sample is None:
        raise gr.Error('Upload a voice sample first.')
    name = (profile_name or '').strip()
    if not name:
        raise gr.Error('Enter a profile name.')
    safe = ''.join(c for c in name if c.isalnum() or c in (' ','_','-')).strip()
    if not safe:
        raise gr.Error('Invalid profile name.')

    src = Path(sample)
    dst = VOICE_BANK / f'{safe}.wav'
    shutil.copy2(src, dst)
    return f'Saved: {dst}'

def generate(text, profile_name):
    # Deliberately fail clearly instead of pretending to clone the voice.
    if not (text or '').strip():
        raise gr.Error('Enter text.')
    profile = VOICE_BANK / f'{profile_name}.wav'
    if not profile.exists():
        raise gr.Error('Voice profile not found.')
    raise gr.Error(
        'Reference-voice generation backend is not configured in this release. '
        'The old MMS-TTS fallback was removed because it did not clone the uploaded voice.'
    )

with gr.Blocks(title='AI Music OS — Voice Profiles') as demo:
    gr.Markdown('## Voice profiles\nUpload and persist a reference sample. Generation is disabled until a real speaker-conditioned backend is configured.')
    sample = gr.Audio(type='filepath', label='Reference voice sample')
    profile = gr.Textbox(label='Profile name')
    save = gr.Button('Save profile')
    status = gr.Textbox(label='Status')
    save.click(save_voice, [sample, profile], status)

    text = gr.Textbox(lines=6, label='Text')
    gen = gr.Button('Generate')
    gen.click(generate, [text, profile], gr.Textbox(label='Output'))

demo.launch(share=True, debug=True)
